# 13 — Collecte Facebook avec Apify

Cette version utilise l’Actor `apify/facebook-posts-scraper` fourni dans l’exemple. Elle ne lance plus Chrome depuis Jupyter. La cible validée est de 30 publications par page ; le test pilote reste limité à 10 posts ENSEA afin de contrôler le résultat et la consommation de crédits Apify.

Les données restent isolées dans `data/facebook/raw/` et ne modifient pas l’index RAG.

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Projet :', PROJECT_ROOT)
print('apify-client installé :', importlib.util.find_spec('apify_client') is not None)

Projet : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA
apify-client installé : True


## Installation

Si la cellule installe `apify-client`, redémarrez ensuite le kernel et reprenez le notebook depuis le début.

In [2]:
if importlib.util.find_spec('apify_client') is None:
    %pip install "apify-client>=1.8"
    print('Installation terminée : redémarrez maintenant le kernel.')
else:
    print('apify-client est déjà disponible.')

apify-client est déjà disponible.


## Jeton Apify

Le jeton est saisi de façon masquée et reste uniquement en mémoire. **Ne remplacez pas cette cellule par un jeton écrit en clair**, car celui-ci finirait dans le fichier notebook.

In [3]:
from getpass import getpass

APIFY_TOKEN = os.getenv('APIFY_TOKEN') or getpass('Jeton Apify : ')
assert APIFY_TOKEN.strip(), 'Le jeton Apify est vide.'
print('Jeton chargé en mémoire, sans affichage ni écriture sur disque.')

Jeton chargé en mémoire, sans affichage ni écriture sur disque.


In [4]:
import importlib
import facebook_scraper as facebook_scraper_module
from config import load_config

facebook_scraper_module = importlib.reload(facebook_scraper_module)
FacebookScraper = facebook_scraper_module.FacebookScraper
config = load_config(PROJECT_ROOT / 'config.yaml')
facebook = config.facebook_scraping
scraper = FacebookScraper(config, api_token=APIFY_TOKEN)

print('Actor :', facebook.actor_id)
print(f'{len(facebook.pages)} pages ; cible finale = {facebook.max_posts_per_page} posts/page')
print('Sortie isolée :', facebook.raw_full_path)

Actor : apify/facebook-posts-scraper
8 pages ; cible finale = 100 posts/page
Sortie isolée : <PROJECT_ROOT>\data\facebook\raw


## Test pilote — un appel Apify, 10 posts ENSEA

Le premier run a déjà réussi côté Apify. `PILOT_REUSE_RUN_ID` permet de récupérer son dataset sans relancer l’Actor et sans consommer de nouveaux crédits. Mettez cette variable à `None` uniquement pour demander une nouvelle collecte.

In [5]:
from dataclasses import asdict

PILOT_PAGE = 'ENSEA_page'
PILOT_MAX_POSTS = 10
PILOT_REUSE_RUN_ID = 'obye1IUTNR0avQKoW'  # run réussi déjà payé
pilot_report = scraper.scrape_page(
    PILOT_PAGE,
    max_posts=PILOT_MAX_POSTS,
    reuse_run_id=PILOT_REUSE_RUN_ID,
)
asdict(pilot_report)

{'page_key': 'ENSEA_page',
 'page_url': 'https://web.facebook.com/ensea.ci',
 'requested_posts': 10,
 'raw_items': 10,
 'unique_posts': 10,
 'skipped_empty_or_short': 0,
 'actor_run_id': 'obye1IUTNR0avQKoW',
 'dataset_id': '8Big9Vt5VEyUerBNT',
 'normalized_path': 'C:\\Users\\Ariel\\OneDrive - ENSEA\\Documents\\LLMs\\RAG ENSEA\\data\\facebook\\raw\\ENSEA_page.jsonl',
 'raw_path': 'C:\\Users\\Ariel\\OneDrive - ENSEA\\Documents\\LLMs\\RAG ENSEA\\data\\facebook\\raw\\ENSEA_page_raw.json',
 'warning': None}

In [6]:
import pandas as pd

pilot_path = Path(pilot_report.normalized_path)
pilot_rows = [json.loads(line) for line in pilot_path.read_text(encoding='utf-8').splitlines() if line.strip()]
pilot_df = pd.DataFrame(pilot_rows)
if pilot_df.empty:
    print('Aucun post textuel normalisé. Rapport :', asdict(pilot_report))
else:
    display(pilot_df[['author', 'published_at', 'post_url', 'likes', 'comments', 'shares', 'text']])
    print('Posts uniques :', pilot_df['post_id'].nunique(), '/', len(pilot_df))
    print('Liens permanents présents :', f"{pilot_df['post_url'].notna().mean():.0%}")
    print('Dates présentes :', f"{pilot_df['published_at'].notna().mean():.0%}")
    print('Longueur médiane :', int(pilot_df['text'].str.len().median()), 'caractères')

,author,published_at,post_url,likes,comments,shares,text
0,ensea.ci,2026-07-27T13:13:46+00:00,https://www.facebook.com/ensea.ci/posts/pfbid0...,196,14,7,𝗣𝗥𝗢𝗖𝗘̀𝗦-𝗩𝗘𝗥𝗕𝗔𝗟 𝗗𝗨 𝗝𝗨𝗥𝗬 𝗗𝗘 𝗥𝗘𝗖𝗥𝗨𝗧𝗘𝗠𝗘𝗡𝗧 𝗗𝗘 𝗖𝗔𝗡𝗗𝗜...
1,ensea.ci,2026-07-27T11:40:30+00:00,https://www.facebook.com/ensea.ci/posts/pfbid0...,49,0,1,#LaMotivDeLaSemaine\n\nNous passons souvent be...
2,ensea.ci,2026-07-22T16:11:42+00:00,https://www.facebook.com/ensea.ci/posts/pfbid0...,93,1,2,📢 𝗡𝗢𝗨𝗩𝗘𝗟 𝗔𝗥𝗧𝗜𝗖𝗟𝗘 𝗔̀ 𝗟𝗜𝗥𝗘 | 𝗡𝗲𝘄𝘀𝗹𝗲𝘁𝘁𝗲𝗿 𝗔𝗠𝗔𝗡𝗜𝗘𝗛 ...
3,ensea.ci,2026-07-20T16:24:41+00:00,https://www.facebook.com/ensea.ci/posts/pfbid0...,242,4,17,𝗥𝗲𝘁𝗼𝘂𝗿 𝘀𝘂𝗿 𝗹𝗮 𝗖𝗼𝗻𝗳𝗲́𝗿𝗲𝗻𝗰𝗲-𝗗𝗲́𝗯𝗮𝘁 𝗘𝗡𝗦𝗘𝗔 𝘅 𝗛𝗮𝗿𝘃𝗮...
4,ensea.ci,2026-07-20T09:23:23+00:00,https://www.facebook.com/ensea.ci/posts/pfbid0...,131,1,2,#LaMotivDeLaSemaine \n\nLa gratitude est bien ...
5,ensea.ci,2026-07-14T11:22:43+00:00,https://www.facebook.com/reel/1410672734298320/,2053,12,77,"Suivez l'interview exclusive de #Gomez_AGOU, d..."
6,ensea.ci,2026-07-13T16:09:53+00:00,https://www.facebook.com/ensea.ci/posts/pfbid0...,27,1,2,𝗘𝗡𝗦𝗘𝗔 | 𝗜𝗡𝗣-𝗛𝗕 : 𝗔𝗣𝗣𝗘𝗟 𝗔̀ 𝗖𝗔𝗡𝗗𝗜𝗗𝗔𝗧𝗨𝗥𝗘\n\nL'Eco...
7,ensea.ci,2026-07-13T16:07:15+00:00,https://www.facebook.com/ensea.ci/posts/pfbid0...,53,1,3,#LaMotivDeLaSemaine\n\nNous passons souvent no...
8,ensea.ci,2026-07-13T09:19:43+00:00,https://www.facebook.com/ensea.ci/posts/pfbid0...,52,9,5,𝗘𝗡𝗦𝗘𝗔 | 𝗠𝗔𝗦𝗧𝗘𝗥 𝗜𝗜 𝗘𝗡 𝗦𝗧𝗔𝗧𝗜𝗦𝗧𝗜𝗤𝗨𝗘𝗦 𝗔𝗚𝗥𝗜𝗖𝗢𝗟𝗘𝗦 : ...
9,ensea.ci,2026-07-10T17:56:55+00:00,https://www.facebook.com/ensea.ci/posts/pfbid0...,26,0,6,𝗘𝗡𝗦𝗘𝗔 | 𝗜𝗡𝗣-𝗛𝗕 : 𝗔𝗣𝗣𝗘𝗟 𝗔̀ 𝗖𝗔𝗡𝗗𝗜𝗗𝗔𝗧𝗨𝗥𝗘\n\nL'Eco...


Posts uniques : 10 / 10
Liens permanents présents : 100%
Dates présentes : 100%
Longueur médiane : 1065 caractères


In [7]:
if not pilot_df.empty:
    assert len(pilot_df) <= PILOT_MAX_POSTS
    assert pilot_df['post_id'].is_unique, 'Des doublons subsistent.'
    assert pilot_df['source'].eq('facebook').all()
    assert pilot_df['page_key'].eq(PILOT_PAGE).all()
    assert pilot_df['text'].str.len().ge(facebook.min_post_characters).all()
    print('Contrôles structurels validés.')
else:
    print('Contrôles non exécutés : inspectez le fichier brut et le rapport.')

Contrôles structurels validés.


## Collecte complète — attendre la validation du pilote

La cellule suivante créera huit exécutions Apify et demandera jusqu’à 240 posts.

In [9]:
RUN_FULL_COLLECTION = True

if RUN_FULL_COLLECTION:
    full_reports = scraper.run(max_posts_per_page=30)
    display(pd.DataFrame([asdict(report) for report in full_reports]))
else:
    print('Collecte complète désactivée : comportement attendu pour le pilote.')

[apify.facebook-posts-scraper runId:Q2kstcE3kTZ09EXSY] -> Status: RUNNING, Message: 
[apify.facebook-posts-scraper runId:Q2kstcE3kTZ09EXSY] -> 2026-08-03T15:45:09.586Z ACTOR: Pulling container image of build 5wceITIpJWPtxIixz from registry.
[apify.facebook-posts-scraper runId:Q2kstcE3kTZ09EXSY] -> 2026-08-03T15:45:09.589Z ACTOR: Creating container.
[apify.facebook-posts-scraper runId:Q2kstcE3kTZ09EXSY] -> 2026-08-03T15:45:09.861Z ACTOR: Starting container.
[apify.facebook-posts-scraper runId:Q2kstcE3kTZ09EXSY] -> 2026-08-03T15:45:09.861Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.facebook-posts-scraper runId:Q2kstcE3kTZ09EXSY] -> 2026-08-03T15:45:10.719Z INFO  System info {"apifyVersion":"3.7.2","apifyClientVersion":"2.23.4","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.18.0"}
[apify.facebook-posts-scraper runId:Q2kstcE3kTZ09EXSY] -> 2026-08-03T15:45:10.807Z (node:7) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors tha

,page_key,page_url,requested_posts,raw_items,unique_posts,skipped_empty_or_short,actor_run_id,dataset_id,normalized_path,raw_path,warning
0,ENSEA_page,https://web.facebook.com/ensea.ci,30,30,30,0,Q2kstcE3kTZ09EXSY,eheHXAyl74jIGkieM,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,NaN
1,CDR_page,https://web.facebook.com/cdr.ensea.1,30,30,30,0,RWxaTTtyVdeY0xdcC,2HTNXcYj0DoFVnp5Z,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,NaN
2,EJS_page,https://web.facebook.com/profile.php?id=100057...,30,30,29,1,hrTQucVy0soUcPQ2Q,w7nddohO7Zkt7m8uS,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,29 posts textuels uniques obtenus sur 30 deman...
3,EDSC_page,https://web.facebook.com/edsc.ensea,30,30,29,1,XnSmeY1jnW9SYMOzj,6lcYOWF6B4nnlofyH,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,29 posts textuels uniques obtenus sur 30 deman...
4,ECOSTAT_page,https://web.facebook.com/DepecheEcostat,30,30,30,0,hTD0v125YA9CqLIxU,hZHPSf6tUa8tVj5yj,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,NaN
5,AES_page,https://web.facebook.com/profile.php?id=615665...,30,30,28,2,LCrDKS9QsZnqT3XhI,O9zYAZNDT1MPSU8zb,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,28 posts textuels uniques obtenus sur 30 deman...
6,DISE_page,https://web.facebook.com/profile.php?id=615772...,30,30,30,0,ByywTXwr4K9DmHxsF,8e236YUzmxxGrxCdJ,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,NaN
7,DAS_page,https://web.facebook.com/DivisionAS,30,30,26,4,VA56JkEZaKSZ9a2JJ,3MOoj9fAlybTnFdvJ,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,C:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs...,26 posts textuels uniques obtenus sur 30 deman...
